In [0]:
import os
account_key = os.environ.get("AZURE_STORAGE_KEY")

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

tableName = "Acx_testdetails"

bronze_path = f"abfss://raw@hisadls.dfs.core.windows.net/bronzewa/{tableName}"
silver_path = f"abfss://raw@hisadls.dfs.core.windows.net/silverwa/{tableName}"

checkpoint_path = f"abfss://raw@hisadls.dfs.core.windows.net/checkpoints/{tableName}"

In [0]:
# bronze_path = f"/Volumes/main/default/raw_volume/bronzewa/{tableName}"
# silver_path = f"/Volumes/main/default/raw_volume/silverwa/{tableName}"
# checkpoint_path = f"/Volumes/main/default/raw_volume/checkpoints/{tableName}"

In [0]:
bronze_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", checkpoint_path + "/schema")
        .load(bronze_path)
)

In [0]:
# merge works even if new columns arrive.

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window = Window.partitionBy("SALESID", "ITEMID").orderBy(desc("BOOKINGDATE"))

def upsert_to_silver(microBatchDF, batchId):
    microBatchDF = microBatchDF.withColumn("rn", row_number().over(window)) \
                               .filter("rn = 1").drop("rn")                            # Deduplicate
    
    if not DeltaTable.isDeltaTable(spark, silver_path):
        microBatchDF.write.format("delta").mode("overwrite").save(silver_path)
    
    deltaTable = DeltaTable.forPath(spark, silver_path)
        
    deltaTable.alias("T") \
        .merge(
            microBatchDF.alias("S"),
            "T.SALESID = S.SALESID"
        ) \
        .whenMatchedUpdate(
            condition="S.BOOKINGDATE > T.BOOKINGDATE",
            set={col: f"S.{col}" for col in microBatchDF.columns
                    if col not in ["SALESID", "ITEMID"]}
        ) \
        .whenNotMatchedInsertAll() \
        .execute()

In [0]:
query = (
    bronze_stream.writeStream
        .foreachBatch(upsert_to_silver)
        .option("checkpointLocation", checkpoint_path)
        .trigger(once=True)                                 # Runs once like batch
        .start()
)

query.awaitTermination()

In [0]:
max_watermark = (
    spark.read.format("delta")
        .load(silver_path)
        .selectExpr("max(BOOKINGDATE) as max_val")
        .collect()[0]["max_val"]
)

dbutils.notebook.exit(str(max_watermark))

##### - this Auto Loader version does NOT trigger unnecessary I/O
##### - Lists new files efficiently
##### - Only processes unprocessed files
##### - Does NOT rescan entire bronze every run

In [0]:
silver_path = f"abfss://raw@hisadls.dfs.core.windows.net/silverwa/Acx_testdetails"

spark.read.format("delta").load(silver_path).count()

11533

In [0]:
bronze_path = f"abfss://raw@hisadls.dfs.core.windows.net/bronzewa/{tableName}"
spark.read.format("parquet").load(bronze_path).count()

34599